# 04 — Feature Engineering

Every target-derived feature is computed **as of the forecast origin**, never as of the target date.

To forecast day `origin+14`, the model may use demand up to `origin`. It may not use a lag measured from the target date, because that would be `origin+13` — a day that has not happened yet. Building `rolling_mean_7` with a panel-wide groupby-shift is the classic way this goes wrong, and it inflates reported accuracy substantially.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

In [ ]:
from src.data.loader import rolling_origin_folds
from src.features.build import build_supervised, origin_features, feature_columns
fold = rolling_origin_folds(cfg, panel)[-1]
df = build_supervised(cfg, panel, fold)
print(f'{len(df):,} rows, {len(feature_columns(df))} features')
print('lags:', cfg.lags, '| rolling windows:', cfg.rolling_windows)
df.head(3)

## Origin-anchored features for one origin

In [ ]:
f = origin_features(panel[panel.date <= fold.origin], fold.origin, cfg)
f[[c for c in f.columns if c.startswith(('lag_','rolling_mean_'))]].head()

## Proof that features cannot see the future

Overwrite every actual after the origin with nonsense, rebuild, and compare.

In [ ]:
from src.data.audit import leakage_probe
leakage_probe(cfg, panel, fold)

## What is allowed to come from the future

`sell_price`, `snap` and the calendar for the **target date**. A retailer sets prices and promotion calendars in advance, so these are genuinely known at forecast time. That is an assumption about the business process, declared in `configs/config.yaml` under `features.assume_known_future`.

In [ ]:
cfg['features']